# MCR Stage B — Labeled rollout corpus for Qwen3.6-35B-A3B

**Goal**: build the training substrate for the composite MCR reward (density + circuit edges + adversary).

**What we collect**:
- 200 SuperGPQA questions × 5 sampled rollouts = **1000 labeled rollouts**
- Per-token residual stream activations at **L11, L17, L23** (fp16)
- Verifier label (correct / wrong) per rollout
- Response text + token IDs

**Split**: SuperGPQA `[250:450]` — disjoint from S4-L0 `[0:100]`, G1 discovery `[100:150]`, G1 validation `[150:250]`.

**Sampling**: `temperature=1.0, top_p=0.95, max_new=2048`. Matches GRPO rollout distribution at training.

**Downstream consumers**:
- Stage D (density reward) — fit p_C / p_W GMMs per layer
- Stage E (circuit edges) — attribution-patch cross-layer co-activation mining
- Stage F (adversarial D) — warm-start discriminator on 800/200 train/val split

**Output**: HF repo `caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b` (private). One shard per question (~160 MB safetensors), plus `manifest.json` and `README.md`.

**Resumable**: checks HF repo on startup, skips question_ids already uploaded.

**Budget**: ~5-6h on RTX PRO 6000 Blackwell 96 GB. ~$7 Colab Pro. Storage: ~32 GB.

**Safe to run in parallel with G2** — separate instance, no shared state.

## Cell 1 — Install (same stack as S4-L0 / G1 / G2)

In [ ]:
import sys, subprocess, os, shutil, site
from pathlib import Path

def _pip(*args, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *args], check=check)

def _check():
    try:
        import transformers
        try:
            from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
        except ImportError:
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
        return 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, transformers.__version__
    except Exception as e:
        return False, f'import-error: {e}'

ok, ver = _check()
print(f'Current: transformers={ver}, qwen3_5_moe={ok}')

TRANSFORMERS_REF = 'main'
SRC_DIR = '/content/transformers_src'

if not ok:
    print(f'\n\u2192 Installing transformers @ {TRANSFORMERS_REF}')
    _pip('install', '-q',
         'accelerate', 'datasets',
         'huggingface_hub==1.5.0',
         'safetensors', 'einops', 'scikit-learn',
         'sentencepiece', 'tokenizers', 'protobuf', 'scipy')
    _pip('uninstall', '-y', 'transformers', check=False)
    _pip('uninstall', '-y', 'transformers', check=False)
    for p in sys.path + site.getsitepackages() + [site.getusersitepackages()]:
        tr = Path(p) / 'transformers'
        if tr.exists():
            shutil.rmtree(tr, ignore_errors=True)
    if os.path.exists(SRC_DIR):
        shutil.rmtree(SRC_DIR)
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/huggingface/transformers.git', SRC_DIR], check=True)
    if TRANSFORMERS_REF != 'main':
        subprocess.run(['git', '-C', SRC_DIR, 'checkout', '--quiet', TRANSFORMERS_REF], check=True)
    _pip('install', '--force-reinstall', '--no-deps', '--no-cache-dir', SRC_DIR)
    _pip('install', '--no-cache-dir', 'flash-linear-attention', 'causal-conv1d', check=False)
    for mod in list(sys.modules):
        if mod.startswith('transformers') or mod.startswith('huggingface_hub'):
            del sys.modules[mod]
    ok, ver = _check()
    print(f'Post-install: transformers={ver}, qwen3_5_moe={ok}')
    if not ok:
        print('\n\u26a0\ufe0f  Restart kernel and re-run this cell.')
        raise SystemExit

# HF auth
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('\u2705 HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

import json, time, random, re, gc
import torch
import torch.nn.functional as F
import numpy as np
from safetensors.torch import save_file as save_safetensors
from huggingface_hub import HfApi, hf_hub_download, list_repo_files, create_repo

WORK_ROOT = Path('/content/mcr_stage_b')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SHARD_DIR = WORK_ROOT / 'shards'
SHARD_DIR.mkdir(exist_ok=True)
print(f'Work root: {WORK_ROOT}')

## Cell 2 — CFG

Split indices: `[250:450]` is disjoint from all prior S4 work.
Rollout settings match GRPO training sampler (temp=1.0, top_p=0.95).
Capture layers chosen from S4-L0 mapping: L11 (early semantic peak), L17 (mid-retrieval peak), L23 (late planning, has trained SAE).

In [ ]:
CFG = dict(
    model_id='Qwen/Qwen3.6-35B-A3B',
    # Dataset
    difficulty_filter=['easy', 'middle'],
    split_start=250,
    n_questions=200,
    seed=42,
    # Rollout
    n_rollouts=5,
    max_new_tokens=2048,
    temperature=1.0,
    top_p=0.95,
    # Capture layers
    capture_layers=[11, 17, 23],
    # Output
    hf_repo='caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
    private_repo=True,
    activation_dtype='float16',
)
print(json.dumps(CFG, indent=2))

## Cell 3 — HF repo setup + resume list

Creates the target repo if missing, then fetches the list of already-uploaded shards so we skip them.
A shard is one question's data: `q{global_idx:04d}.safetensors`.

In [ ]:
api = HfApi()
try:
    create_repo(CFG['hf_repo'], repo_type='dataset', private=CFG['private_repo'], exist_ok=True)
    print(f'\u2705 HF dataset repo ready: {CFG["hf_repo"]}')
except Exception as e:
    print(f'(repo exists or error: {e})')

existing = set()
try:
    files = list_repo_files(CFG['hf_repo'], repo_type='dataset')
    for f in files:
        m = re.match(r'shards/q(\d{4})\.safetensors', f)
        if m:
            existing.add(int(m.group(1)))
    print(f'Already uploaded: {len(existing)} shards')
    if existing:
        print(f'  range: [{min(existing)}, {max(existing)}]')
except Exception as e:
    print(f'(no existing files: {e})')

resume_from = CFG['split_start'] + len(existing)  # informational only
print(f'\nResume: will skip {len(existing)} questions, ~{CFG["n_questions"] - len(existing)} remaining')

## Cell 4 — Load Qwen3.6-35B-A3B (bf16, SDPA)

In [ ]:
import transformers
try:
    from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
except ImportError:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
assert 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, 'Restart kernel, rerun Cell 1.'

from transformers import AutoTokenizer, AutoModelForImageTextToText

if 'model' not in dir() or not hasattr(globals().get('model'), 'named_parameters'):
    tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForImageTextToText.from_pretrained(
        CFG['model_id'],
        dtype=torch.bfloat16,
        device_map='cuda',
        attn_implementation='sdpa',
        trust_remote_code=True,
    )
    model.eval()
    print(f'Model loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')
else:
    print('Reusing existing model in kernel.')

if 'tok' not in dir():
    tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token

gc.collect(); torch.cuda.empty_cache()
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

## Cell 5 — SuperGPQA dataset slice

Indices `[250:450]`, stratified by discipline (best effort via the shuffled order).

In [ ]:
from datasets import load_dataset

raw = load_dataset('m-a-p/SuperGPQA', split='train')
filtered = raw.filter(lambda ex: ex['difficulty'] in CFG['difficulty_filter'])
shuffled = filtered.shuffle(seed=CFG['seed'])
end = CFG['split_start'] + CFG['n_questions']
assert len(shuffled) >= end, f'dataset too small: {len(shuffled)} < {end}'

stage_b = shuffled.select(range(CFG['split_start'], end))
print(f'Stage B slice: {len(stage_b)} questions (indices [{CFG["split_start"]}:{end}])')

# Discipline distribution
from collections import Counter
disc_counts = Counter(ex['discipline'] for ex in stage_b)
for d, c in sorted(disc_counts.items(), key=lambda kv: -kv[1]):
    print(f'  {d:30s} {c:3d}')

## Cell 6 — Prompt template + MCQ verifier (identical to G1/G2)

In [ ]:
def format_prompt(ex):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(ex['options']))
    msgs = [{
        'role': 'user',
        'content': (
            f"Answer the following multiple-choice question. Reason step by step, "
            f"then put the letter of the correct answer in \\boxed{{}}.\n\n"
            f"Question: {ex['question']}\n\n"
            f"Options:\n{choices}"
        )
    }]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

BOXED_RE = re.compile(r'\\boxed\{\s*([A-J])\s*\}')
LETTER_AT_END_RE = re.compile(r'[Aa]nswer[:\s]+([A-J])\b')

def extract_letter(completion, n_opts):
    m = list(BOXED_RE.finditer(completion))
    if m:
        letter = m[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts: return letter
    m2 = list(LETTER_AT_END_RE.finditer(completion))
    if m2:
        letter = m2[-1].group(1).upper()
        if ord(letter) - ord('A') < n_opts: return letter
    tail = completion[-200:]
    cands = re.findall(r'\b([A-J])\b', tail)
    if cands:
        letter = cands[-1]
        if ord(letter) - ord('A') < n_opts: return letter
    return None

def verify(ex, completion):
    pred = extract_letter(completion, len(ex['options']))
    return pred is not None and pred == ex['answer_letter']

## Cell 7 — Multi-layer hooks (L11, L17, L23)

Uses the same layer locator as S4-L0/G1. Forward hook captures the full `[B, T, d_model]` hidden tensor at each target layer. We trigger hooks by running a single second forward on the generated batch (clean, no KV-cache ambiguity).

In [ ]:
def get_layer_module(m, idx):
    candidates = [m]
    if hasattr(m, 'base_model') and m.base_model is not m:
        candidates.append(m.base_model.model if hasattr(m.base_model, 'model') else m.base_model)
    if hasattr(m, 'model'):
        candidates.append(m.model)
    paths = [
        ('model', 'language_model', 'layers'),
        ('language_model', 'layers'),
        ('model', 'layers'),
        ('layers',),
    ]
    for start in candidates:
        for path in paths:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except (IndexError, TypeError): continue
    raise RuntimeError(f'Could not locate layer {idx}')

class MultiCapture:
    """Holds hidden states from multiple layers. h[layer_idx] = [B, T, d_model]."""
    def __init__(self, layer_indices):
        self.h = {i: None for i in layer_indices}
    def make_hook(self, layer_idx):
        def hook(module, inp, out):
            h = out[0] if isinstance(out, tuple) else out
            self.h[layer_idx] = h.detach()
        return hook
    def clear(self):
        for k in self.h:
            self.h[k] = None

def register_multi_capture(m, layer_indices):
    cap = MultiCapture(layer_indices)
    handles = []
    for idx in layer_indices:
        h = get_layer_module(m, idx).register_forward_hook(cap.make_hook(idx))
        handles.append(h)
    return cap, handles

# Probe
cap, handles = register_multi_capture(model, CFG['capture_layers'])
with torch.inference_mode():
    probe_in = torch.randint(0, 1000, (2, 16), device='cuda')
    _ = model(input_ids=probe_in, attention_mask=torch.ones_like(probe_in), use_cache=False)
for li in CFG['capture_layers']:
    assert cap.h[li] is not None, f'Hook at L{li} did not fire'
    print(f'L{li}: {tuple(cap.h[li].shape)}  dtype={cap.h[li].dtype}')
d_model = cap.h[CFG['capture_layers'][0]].shape[-1]
print(f'\nd_model = {d_model}')
for h in handles: h.remove()
cap.clear()
gc.collect(); torch.cuda.empty_cache()

## Cell 8 — Per-rollout extraction utilities

Given generated batch `[n_rollouts, T]`, returns:
- `response_lens`: list of int, response-only length per rollout (prompt stripped, trailing padding stripped)
- `acts_per_layer`: dict[layer_idx] → list[tensor [L, d_model] fp16] (one per rollout, response-only)
- `completions`: list[str]
- `outcomes`: list[bool]

In [ ]:
def compute_response_lens(gen, prompt_len, pad_id, eos_id):
    """For each row in `gen [B, T]`, return response length (tokens after prompt, before first EOS or pad)."""
    B, T = gen.shape
    lens = []
    for b in range(B):
        row = gen[b]
        resp = row[prompt_len:]
        L = 0
        for tid in resp.tolist():
            if tid == eos_id or tid == pad_id:
                break
            L += 1
        # Include EOS token if present (standard for output-scoring)
        if L < len(resp) and resp[L].item() == eos_id:
            L += 1
        lens.append(L)
    return lens

def slice_response_acts(h_full, prompt_len, response_lens):
    """h_full: [B, T, d_model]. Returns list of [L_i, d_model] per rollout (response tokens only)."""
    out = []
    for b, L in enumerate(response_lens):
        if L == 0:
            out.append(torch.zeros((0, h_full.shape[-1]), dtype=torch.float16))
        else:
            out.append(h_full[b, prompt_len:prompt_len+L].to(torch.float16).cpu())
    return out

print('Extraction utilities ready.')

## Cell 9 — Main collection loop

Per question:
1. Generate 5 rollouts in one batched `generate()` (`num_return_sequences=5`, temp=1.0, top_p=0.95)
2. Second forward on the full `[5, T_max]` with 3 hooks attached → captures L11/L17/L23 hidden states
3. Per-rollout: slice response-only activations, convert to fp16
4. Verify each rollout against gold
5. Save shard `q{idx:04d}.safetensors` → upload to HF → delete local

Progress: logs every question with timing + per-question accuracy.

In [ ]:
from tqdm.auto import tqdm

model.config.use_cache = True
pad_id = tok.pad_token_id
eos_id = tok.eos_token_id

# Persistent hook registration for the whole loop
cap, handles = register_multi_capture(model, CFG['capture_layers'])

total_correct = 0
total_rollouts = 0
start_time = time.time()

try:
    for q_i, ex in enumerate(tqdm(stage_b, desc='Stage B')):
        global_idx = CFG['split_start'] + q_i
        if global_idx in existing:
            continue
        q_t0 = time.time()

        prompt = format_prompt(ex)
        enc = tok(prompt, return_tensors='pt').to('cuda')
        P = enc.input_ids.shape[1]

        # Step 1: generate 5 rollouts, sampling
        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=CFG['max_new_tokens'],
                do_sample=True,
                temperature=CFG['temperature'],
                top_p=CFG['top_p'],
                num_return_sequences=CFG['n_rollouts'],
                pad_token_id=pad_id,
                use_cache=True,
            )
        # gen: [n_rollouts, T]
        response_lens = compute_response_lens(gen, P, pad_id, eos_id)

        # Step 2: second forward for clean hidden states (no KV cache, hooks fire on full seq)
        cap.clear()
        attn_mask = (gen != pad_id).long()
        with torch.no_grad():
            model(input_ids=gen, attention_mask=attn_mask, use_cache=False)

        # Step 3: slice response-only activations per rollout, per layer
        acts_per_layer = {}
        for li in CFG['capture_layers']:
            acts_per_layer[li] = slice_response_acts(cap.h[li], P, response_lens)
        cap.clear()

        # Step 4: verify + decode
        q_correct = 0
        rollout_records = []
        for r in range(CFG['n_rollouts']):
            resp_tokens = gen[r, P:P+response_lens[r]].cpu().tolist()
            completion = tok.decode(resp_tokens, skip_special_tokens=True)
            is_correct = verify(ex, completion)
            pred = extract_letter(completion, len(ex['options']))
            rollout_records.append({
                'rollout_id': r,
                'response_text': completion,
                'response_tokens': resp_tokens,
                'response_len': response_lens[r],
                'pred': pred,
                'correct': bool(is_correct),
            })
            q_correct += int(is_correct)
        total_correct += q_correct
        total_rollouts += CFG['n_rollouts']

        # Step 5: pack activations + metadata into a safetensors shard
        # Layout: concatenate all rollouts' acts per layer into one contiguous tensor,
        # keep offsets so downstream can slice.
        shard_tensors = {}
        offsets = {li: [0] for li in CFG['capture_layers']}
        for li in CFG['capture_layers']:
            rolls = acts_per_layer[li]  # list of [L_r, d_model] fp16
            for t in rolls:
                offsets[li].append(offsets[li][-1] + t.shape[0])
            cat = torch.cat(rolls, dim=0) if rolls else torch.zeros((0, d_model), dtype=torch.float16)
            shard_tensors[f'acts_L{li}'] = cat.contiguous()
        shard_tensors['outcomes'] = torch.tensor([r['correct'] for r in rollout_records], dtype=torch.uint8)
        shard_tensors['response_lens'] = torch.tensor([r['response_len'] for r in rollout_records], dtype=torch.int32)

        shard_meta = {
            'question_global_idx': str(global_idx),
            'question': ex['question'][:4000],
            'options': json.dumps(ex['options']),
            'gold_letter': ex['answer_letter'],
            'discipline': ex.get('discipline', ''),
            'field': ex.get('field', ''),
            'difficulty': ex.get('difficulty', ''),
            'prompt_len': str(P),
            'n_rollouts': str(CFG['n_rollouts']),
            'capture_layers': json.dumps(CFG['capture_layers']),
            'offsets': json.dumps({f'L{li}': offsets[li] for li in CFG['capture_layers']}),
            'rollouts': json.dumps([
                {k: r[k] for k in ('rollout_id', 'response_text', 'response_len', 'pred', 'correct')}
                for r in rollout_records
            ]),
            'response_tokens': json.dumps([r['response_tokens'] for r in rollout_records]),
        }

        shard_path = SHARD_DIR / f'q{global_idx:04d}.safetensors'
        save_safetensors(shard_tensors, str(shard_path), metadata=shard_meta)
        shard_size_mb = shard_path.stat().st_size / 1e6

        # Upload to HF
        try:
            api.upload_file(
                path_or_fileobj=str(shard_path),
                path_in_repo=f'shards/q{global_idx:04d}.safetensors',
                repo_id=CFG['hf_repo'],
                repo_type='dataset',
            )
            shard_path.unlink()
            upload_note = 'uploaded'
        except Exception as e:
            upload_note = f'UPLOAD_FAIL: {e}'

        q_dt = time.time() - q_t0
        tok_count = sum(response_lens)
        running_acc = 100 * total_correct / max(total_rollouts, 1)
        elapsed = (time.time() - start_time) / 60
        print(f'[q{global_idx:04d}] {q_correct}/{CFG["n_rollouts"]} correct | '
              f'tok={tok_count} | size={shard_size_mb:.0f}MB | {q_dt:.0f}s | '
              f'running acc={running_acc:.1f}% | elapsed={elapsed:.1f}min | {upload_note}')

        # Free GPU memory aggressively between questions
        del gen, attn_mask, acts_per_layer, shard_tensors
        gc.collect(); torch.cuda.empty_cache()
finally:
    for h in handles: h.remove()

print(f'\n{"="*60}')
print(f'Stage B collection complete')
print(f'{"="*60}')
print(f'Total rollouts: {total_rollouts}')
print(f'Total correct: {total_correct} ({100*total_correct/max(total_rollouts,1):.1f}%)')
print(f'Wall time: {(time.time()-start_time)/60:.1f} min')

## Cell 10 — Build + upload `manifest.json` and dataset `README.md`

Manifest lists every shard, its discipline + outcome stats, and a schema hint for downstream loaders.
README is the public-facing description of what the dataset contains and how to use it.

In [ ]:
# Re-list to get fresh state
files = list_repo_files(CFG['hf_repo'], repo_type='dataset')
shards = sorted([f for f in files if re.match(r'shards/q\d{4}\.safetensors$', f)])
print(f'Shards in repo: {len(shards)}')

# Pull per-shard metadata (small; just headers) — aggregate accuracy by discipline, layer-capture coverage
from safetensors import safe_open

per_shard_summary = []
by_discipline = {}
for shard in tqdm(shards, desc='Aggregating'):
    local = hf_hub_download(CFG['hf_repo'], shard, repo_type='dataset', force_download=False)
    with safe_open(local, framework='pt') as f:
        meta = f.metadata() or {}
        rollouts = json.loads(meta.get('rollouts', '[]'))
        n_correct = sum(1 for r in rollouts if r['correct'])
        disc = meta.get('discipline', 'unknown')
        per_shard_summary.append({
            'shard': shard,
            'global_idx': int(meta.get('question_global_idx', '-1')),
            'discipline': disc,
            'difficulty': meta.get('difficulty', ''),
            'n_rollouts': len(rollouts),
            'n_correct': n_correct,
        })
        if disc not in by_discipline:
            by_discipline[disc] = {'rollouts': 0, 'correct': 0}
        by_discipline[disc]['rollouts'] += len(rollouts)
        by_discipline[disc]['correct'] += n_correct

total_r = sum(s['n_rollouts'] for s in per_shard_summary)
total_c = sum(s['n_correct'] for s in per_shard_summary)

manifest = {
    'dataset': CFG['hf_repo'],
    'version': '1.0',
    'created': time.strftime('%Y-%m-%d'),
    'source_model': CFG['model_id'],
    'source_benchmark': 'm-a-p/SuperGPQA',
    'split_indices': f'[{CFG["split_start"]}:{CFG["split_start"]+CFG["n_questions"]}]',
    'difficulty_filter': CFG['difficulty_filter'],
    'seed': CFG['seed'],
    'rollout_settings': {
        'n_rollouts_per_question': CFG['n_rollouts'],
        'max_new_tokens': CFG['max_new_tokens'],
        'temperature': CFG['temperature'],
        'top_p': CFG['top_p'],
        'do_sample': True,
    },
    'capture': {
        'layers': CFG['capture_layers'],
        'd_model': int(d_model),
        'activation_dtype': CFG['activation_dtype'],
        'scope': 'response tokens only (prompt stripped)',
    },
    'schema_per_shard': {
            'tensors': {
            'acts_L{layer}': '[total_resp_tokens, d_model] fp16',
            'outcomes': '[n_rollouts] uint8',
            'response_lens': '[n_rollouts] int32',
        },
        'metadata_json_keys': [
            'question_global_idx', 'question', 'options', 'gold_letter',
            'discipline', 'field', 'difficulty', 'prompt_len',
            'n_rollouts', 'capture_layers', 'offsets', 'rollouts', 'response_tokens',
        ],
        'offsets_note': 'offsets[L11] = [0, L0, L0+L1, ..., L0+L1+...+L_{n_rollouts-1}]; use to slice per-rollout activations',
    },
    'statistics': {
        'n_shards': len(shards),
        'total_rollouts': total_r,
        'total_correct': total_c,
        'accuracy': round(total_c / max(total_r, 1), 4),
        'by_discipline': {
            d: {
                'rollouts': v['rollouts'],
                'correct': v['correct'],
                'accuracy': round(v['correct']/max(v['rollouts'],1), 4),
            } for d, v in sorted(by_discipline.items())
        },
    },
    'downstream_consumers': {
        'density_gmm_fit': 'Stage D — fit p_C / p_W GMM(K=5) in PCA-64 space per layer',
        'circuit_edge_mining': 'Stage E — cross-layer attribution patching on correct-labeled responses',
        'adversary_warmstart': 'Stage F — train MLP(3x384→128→64→1) on 80/20 split, target warm AUROC > 0.75',
    },
}

manifest_path = WORK_ROOT / 'manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
api.upload_file(
    path_or_fileobj=str(manifest_path),
    path_in_repo='manifest.json',
    repo_id=CFG['hf_repo'],
    repo_type='dataset',
)
print(f'\u2705 manifest uploaded')
print(json.dumps(manifest['statistics'], indent=2))

In [ ]:
readme = f"""---
license: cc-by-nc-4.0
pretty_name: Qwen3.6-35B-A3B MCR Stage-B Rollout Corpus
tags:
  - sparse-autoencoder
  - reinforcement-learning
  - mechanistic-interpretability
  - reward-modeling
size_categories:
  - 1K<n<10K
---

# Qwen3.6-35B-A3B — MCR Stage-B Labeled Rollout Corpus

Labeled rollout corpus for training the composite **Mechanistic Circuit Reward (MCR)** used in the mechreward project.

## Contents

- **{manifest['statistics']['n_shards']} questions** from SuperGPQA ({', '.join(CFG['difficulty_filter'])} difficulty), indices `[{CFG['split_start']}:{CFG['split_start']+CFG['n_questions']}]` of the seed-{CFG['seed']} shuffled filtered set
- **{manifest['statistics']['total_rollouts']} rollouts** ({CFG['n_rollouts']} per question, sampled at T={CFG['temperature']}, top_p={CFG['top_p']}, max_new={CFG['max_new_tokens']})
- **Overall accuracy**: {manifest['statistics']['accuracy']:.1%}
- **Per-token residual-stream activations** at layers {CFG['capture_layers']} (response tokens only, fp16)

Disjoint from the S4-L0 probe set (`[0:100]`) and G1 discovery/validation (`[100:250]`).

## Shard schema

Each `shards/q{{global_idx:04d}}.safetensors` contains:

| Tensor | Shape | Dtype | Description |
|---|---|---|---|
| `acts_L11` | `[sum(L_r), d_model]` | fp16 | Concatenated L11 residuals, all rollouts |
| `acts_L17` | `[sum(L_r), d_model]` | fp16 | Concatenated L17 residuals |
| `acts_L23` | `[sum(L_r), d_model]` | fp16 | Concatenated L23 residuals |
| `outcomes` | `[n_rollouts]` | uint8 | 1 = correct, 0 = wrong |
| `response_lens` | `[n_rollouts]` | int32 | Response-only token count per rollout |

`metadata['offsets']` gives cumulative row offsets per layer so you can slice per-rollout.

## Loading

```python
from safetensors import safe_open
from huggingface_hub import hf_hub_download
import json

path = hf_hub_download('{CFG['hf_repo']}', 'shards/q0250.safetensors', repo_type='dataset')
with safe_open(path, framework='pt') as f:
    meta = f.metadata()
    offsets = json.loads(meta['offsets'])
    acts_l23 = f.get_tensor('acts_L23')
    outcomes = f.get_tensor('outcomes')
# Rollout r activations at L23: acts_l23[offsets['L23'][r]:offsets['L23'][r+1]]
```

## Intended use

- **Density reward (Stage D)**: fit `p_C^ℓ` / `p_W^ℓ` GMMs per layer in PCA-64 space
- **Circuit-edge mining (Stage E)**: cross-layer attribution patching over correct responses
- **Adversary warm-start (Stage F)**: train `MLP(3×384 → 128 → 64 → 1)` on 80/20 split

## Citation

Companion to the mechreward repo: [github.com/caiovicentino/mechreward](https://github.com/caiovicentino/mechreward).
"""

readme_path = WORK_ROOT / 'README.md'
with open(readme_path, 'w') as f:
    f.write(readme)
api.upload_file(
    path_or_fileobj=str(readme_path),
    path_in_repo='README.md',
    repo_id=CFG['hf_repo'],
    repo_type='dataset',
)
print('\u2705 README uploaded')

## Cell 11 — Sanity checks

Sample 3 random shards, verify tensor shapes align with metadata offsets, and spot-check that correct/wrong labels line up with prediction/gold.

In [ ]:
import random as _random
_random.seed(0)
sample = _random.sample(shards, min(3, len(shards)))
for shard in sample:
    local = hf_hub_download(CFG['hf_repo'], shard, repo_type='dataset', force_download=False)
    with safe_open(local, framework='pt') as f:
        meta = f.metadata()
        offsets = json.loads(meta['offsets'])
        rollouts = json.loads(meta['rollouts'])
        outcomes = f.get_tensor('outcomes').tolist()
        response_lens = f.get_tensor('response_lens').tolist()
        for li in CFG['capture_layers']:
            acts = f.get_tensor(f'acts_L{li}')
            expected_rows = sum(response_lens)
            assert acts.shape[0] == expected_rows, f'{shard} L{li}: rows {acts.shape[0]} != sum(resp_lens) {expected_rows}'
            assert acts.shape[1] == d_model, f'{shard} L{li}: d_model mismatch'
            assert len(offsets[f'L{li}']) == CFG['n_rollouts'] + 1, 'offsets length wrong'
            assert offsets[f'L{li}'][-1] == expected_rows, 'offsets end != total rows'
        # Label-prediction consistency
        for r, oc in zip(rollouts, outcomes):
            gold = meta['gold_letter']
            recomputed = int(r['pred'] == gold) if r['pred'] else 0
            assert recomputed == oc, f'{shard} rollout {r["rollout_id"]}: label {oc} != recomputed {recomputed} (pred={r["pred"]}, gold={gold})'
    print(f'\u2705 {shard}: {sum(outcomes)}/{len(outcomes)} correct, rows per layer = {expected_rows}')
print('\nAll sanity checks passed.')

## Done

Corpus is live at `https://huggingface.co/datasets/{CFG["hf_repo"]}`.

Next stages (run in order or in parallel where independent):
- **Stage C** — train SAEs at L11 + L17 on general text (~$60, 2 weeks)
- **Stage D** — fit density GMMs per layer using this corpus (~1 day, no GPU)
- **Stage E** — circuit-edge attribution patching using this corpus (needs GPU, ~1 week)
- **Stage F** — discriminator warm-start (~1 day, small MLP)

Budget check: Stage B spent ~$7-10 of the ~$490 total budgeted.